In [ ]:
# @title
from IPython.display import display, HTML

display(HTML("""
<script>
const firstCell = document.querySelector('.cell.code_cell');
if (firstCell) {
  firstCell.querySelector('.input').style.pointerEvents = 'none';
  firstCell.querySelector('.input').style.opacity = '0.5';
}
</script>
"""))

html = """
<div style="display:flex; flex-direction:column; align-items:center; text-align:center; gap:12px; padding:8px;">
  <h1 style="margin:0;">&#128075; Welcome to <span style="color:#1E88E5;">Algopath Coding Academy</span>!</h1>

  <img src="https://raw.githubusercontent.com/sshariqali/mnist_pretrained_model/main/algopath_logo.jpg"
       alt="Algopath Coding Academy Logo"
       width="400"
       style="border-radius:15px; box-shadow:0 4px 12px rgba(0,0,0,0.2); max-width:100%; height:auto;" />

  <p style="font-size:16px; margin:0;">
    <em>Empowering young minds to think creatively, code intelligently, and build the future with AI.</em>
  </p>
</div>
"""

display(HTML(html))

## **Part 1: The Assembly Line – Data Preprocessing & Pipeline (Character-Level Seq-to-One)**

**Objective**
Build a robust data preprocessing pipeline for character-level sequence data using a **seq-to-one** approach. In Part 1, you will:

- Learn how to tokenize raw text **at the character level** and build vocabulary mappings (char ↔ index)
- Create input-target pairs where we predict **only the next character** (not the entire sequence)
- Handle variable-length sequences using padding and attention masks
- Construct efficient PyTorch Datasets and DataLoaders for batching and shuffling
---

### **1. Introduction to Character-Level Sequence Data**

**What is Character-Level Modelling?**

Instead of treating words as our basic units, **character-level models** treat individual characters as tokens. This means:
- Each letter, space, punctuation mark, and newline is its own token
- The vocabulary is tiny (only ~60–80 unique characters for English text) compared to word-level models (thousands of words)
- The model must learn to spell words, then construct grammar, then meaning — all from scratch!

**Our Goal: Next Character Prediction (Seq-to-One)**

We want to build a model that predicts **one character** given the previous n characters. Mathematically:

$$P(c_t | c_{t-n}, c_{t-n+1}, ..., c_{t-1})$$

Where:
- $c_t$ is the **single character** we want to predict
- $c_{t-n}, ..., c_{t-1}$ are the previous n characters (context window)

For example:
- **Input:** `"To be or not "` (14 characters)
- **Output:** `"t"` (just the next character)

**Why Character-Level?**
- ✅ Tiny vocabulary (no out-of-vocabulary problem)
- ✅ Can generate any word, including names and made-up words
- ✅ Simpler tokenization — no need for special rules
- ⚠️ Sequences are much longer (characters vs words)
- ⚠️ Model must learn more from less signal per token

### **2. The Dataset: Tiny Shakespeare**

We'll use a collection of Shakespeare's works as our training data. At the character level, this dataset is excellent because:
- It's small enough to train quickly
- It has consistent style — the model must learn poetic spelling and structure
- It demonstrates how character patterns build into words, then sentences

Let's start by loading and exploring the data!

In [ ]:
# Import necessary libraries
import time
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import urllib.request

# Check PyTorch version and device
print(f"PyTorch version: {torch.__version__}")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("GPU not available, using CPU")

In [ ]:
# Download the Tiny Shakespeare dataset
url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
filename = 'tinyshakespeare.txt'

try:
    urllib.request.urlretrieve(url, filename)
    print(f"✓ Downloaded {filename}")
except Exception as e:
    print(f"Error downloading file: {e}")

# Load the text data
with open(filename, 'r', encoding='utf-8') as f:
    text = f.read()

print(f"\n✓ Loaded text data")
print(f"Total characters: {len(text):,}")
print(f"\nFirst 500 characters:")
print("-" * 80)
print(text[:500])
print("-" * 80)

**Character Tokenization: The Simplest Possible Tokenizer**

Unlike word-level tokenization (which required splitting on punctuation and whitespace), character-level tokenization is trivial:
1. Each character in the text becomes one token
2. Spaces, newlines, punctuation — all are their own tokens
3. No preprocessing rules needed!

**Building the Character Vocabulary**

The vocabulary is our mapping from characters to unique integers:
- **char → index** (`char2idx`): Used during training to convert characters to numbers
- **index → char** (`idx2char`): Used during generation to convert numbers back to characters

We still add three special tokens:
- `<PAD>`: Padding token (for making sequences the same length)
- `<UNK>`: Unknown token (for characters not in our vocabulary)
- `<EOS>`: End of sequence token

The key difference: instead of thousands of words, we only have ~65 unique characters!

In [ ]:
# Character tokenization: simply split text into individual characters
def char_tokenize(text):
    """
    Character-level tokenization.
    Each character (including spaces, newlines, punctuation) is its own token.
    """
    return list(text)

# Tokenize the entire text
tokens = char_tokenize(text)
print(f"Total tokens (characters): {len(tokens):,}")
print(f"Unique tokens (vocabulary): {len(set(tokens)):,}")
print(f"\nAll unique characters in the dataset:")
print(repr(''.join(sorted(set(tokens)))))
print(f"\nFirst 50 tokens (characters):")
print(tokens[:50])

In [ ]:
# Build vocabulary: character to index and index to character mappings
def build_vocabulary(tokens, max_vocab_size):
    """
    Build vocabulary from tokens, keeping only the most frequent tokens.

    Args:
        tokens: List of tokens (characters)
        max_vocab_size: Maximum vocabulary size (excluding special tokens)

    Returns:
        char2idx: Dictionary mapping characters to indices
        idx2char: Dictionary mapping indices to characters
    """
    # Count character frequencies
    char_counts = Counter(tokens)

    # Get most common characters
    most_common = char_counts.most_common(max_vocab_size)

    # Create vocabulary with special tokens
    vocab = ['<PAD>', '<UNK>', '<EOS>'] + [char for char, _ in most_common]

    # Create mappings
    char2idx = {char: idx for idx, char in enumerate(vocab)}
    idx2char = {idx: char for char, idx in char2idx.items()}

    return char2idx, idx2char

# Build vocabulary (keep all characters)
char2idx, idx2char = build_vocabulary(tokens, max_vocab_size=len(set(tokens)))
vocab_size = len(char2idx)

print(f"Vocabulary size: {vocab_size:,}")
print(f"\nSpecial tokens:")
print(f"  <PAD> → {char2idx['<PAD>']}")
print(f"  <UNK> → {char2idx['<UNK>']}")
print(f"  <EOS> → {char2idx['<EOS>']}")
print(f"\nFull character vocabulary (after special tokens):")
print({i: repr(idx2char[i]) for i in range(3, min(3 + vocab_size, 3 + 30))})

In [ ]:
# Convert tokens to indices
def tokens_to_indices(tokens, char2idx):
    """
    Convert a list of character tokens to a list of indices.
    Unknown characters are mapped to <UNK>.
    """
    return [char2idx.get(token, char2idx['<UNK>']) for token in tokens]

# Convert our tokens to indices
token_indices = tokens_to_indices(tokens, char2idx)

print(f"Total token indices: {len(token_indices):,}")
print(f"\nExample conversion:")
print(f"Chars:   {tokens[:20]}")
print(f"Indices: {token_indices[:20]}")
print(f"\nVerifying back-conversion:")
print(f"Back to chars: {[idx2char[idx] for idx in token_indices[:20]]}")
print(f"\nAs string: '{''.join([idx2char[idx] for idx in token_indices[:100]])}'")

### **3. The Custom Dataset Class**

**The Sliding Window Approach (Character-Level Seq-to-One)**

We create input-target pairs by sliding a window of `seq_length` characters across the text. Each window predicts just **one next character**:

```
Full text: "To be or not to"
Window size: 10

Window 1:
  Input:  "To be or " (10 chars)
  Target: "n"          (1 char - the 11th character)

Window 2:
  Input:  "o be or no" (10 chars)
  Target: "t"          (1 char - the 12th character)

Window 3:
  Input:  " be or not" (10 chars)
  Target: " "          (1 char - the 13th character)
...
```

Notice:
- The **input** is a sequence of `n` characters
- The **target** is just **one character** (the next one)
- Because characters are finer-grained than words, we use a **longer window** (e.g., seq_length = 100)

In [ ]:
class ShakespeareCharDataset(Dataset):
    """
    Custom Dataset for the Tiny Shakespeare text using Character-Level Seq-to-One approach.
    Creates input-target pairs where target is just the next single character.
    """

    def __init__(self, token_indices, seq_length):
        """
        Initialize the dataset.

        Args:
            token_indices: List of character token indices
            seq_length: Length of each input sequence (window size in characters)
        """
        self.token_indices = token_indices
        self.seq_length = seq_length

        # Calculate how many sequences we can create
        # We need seq_length + 1 characters for each sample (input + 1 target)
        self.num_sequences = len(token_indices) - seq_length

    def __len__(self):
        """Return the total number of sequences in the dataset."""
        return self.num_sequences

    def __getitem__(self, idx):
        """
        Get a single input-target pair (char-level seq-to-one).

        Args:
            idx: Index of the sequence to retrieve

        Returns:
            input_seq: Tensor of input char indices (length = seq_length)
            target: Single target char index (the next character after input_seq)
        """
        # Extract a window of seq_length + 1 characters
        sequence = self.token_indices[idx:idx + self.seq_length + 1]

        # Split into input and target
        input_seq = torch.tensor(sequence[:-1], dtype=torch.long)  # First seq_length chars
        target = torch.tensor(sequence[-1], dtype=torch.long)       # Just the last char

        return input_seq, target

# Create dataset with sequence length of 100 characters
# (Longer than word-level because characters carry less information per token)
seq_length = 100
dataset = ShakespeareCharDataset(token_indices, seq_length)

print(f"Dataset created with Character-Level Seq-to-One approach!")
print(f"Sequence length: {seq_length} characters")
print(f"Total sequences: {len(dataset):,}")
print(f"\nEach sequence consists of:")
print(f"  - Input: {seq_length} characters")
print(f"  - Target: 1 character (just the next one)")

In [ ]:
# Let's examine a few samples to understand the char-level seq-to-one approach
print("Example sequences from the dataset (Character-Level Seq-to-One):\n")

for i in range(3):
    input_seq, target = dataset[i]

    # Convert indices back to characters for visualization
    input_chars = [idx2char[idx.item()] for idx in input_seq]
    target_char = idx2char[target.item()]

    print(f"Sample {i+1}:")
    print(f"  Input:  {''.join(input_chars)!r}")
    print(f"  Target: {target_char!r}  ← Just one character!")
    print(f"  Input indices:  {input_seq.tolist()[:10]}... (first 10)")
    print(f"  Target index: {target.item()}")
    print()

### **4. Handling Variable Lengths: The collate_fn**

**Padding for Variable-Length Character Sequences**

Even at the character level, sequences in a batch may have different lengths in real-world scenarios. Our `collate_fn` handles this by:
1. Finding the longest input sequence in the batch
2. Padding shorter sequences with `<PAD>` (index 0)
3. Creating attention masks (1 = real character, 0 = padding)
4. Stacking single-character targets (no padding needed!)

In [ ]:
def collate_fn(batch):
    """
    Custom collate function for character-level seq-to-one approach.
    Pads input sequences; targets are single char values (no padding needed).

    Args:
        batch: List of tuples (input_seq, target)

    Returns:
        padded_inputs: Tensor of shape (batch_size, max_length)
        targets: Tensor of shape (batch_size,) - single target char per sample
        attention_mask: Tensor of shape (batch_size, max_length)
    """
    inputs, targets = zip(*batch)

    max_length = max(len(seq) for seq in inputs)

    padded_inputs = []
    attention_masks = []

    for input_seq in inputs:
        seq_length = len(input_seq)
        padding_length = max_length - seq_length

        padded_input = torch.cat([input_seq, torch.zeros(padding_length, dtype=torch.long)])
        mask = torch.cat([torch.ones(seq_length, dtype=torch.long),
                          torch.zeros(padding_length, dtype=torch.long)])

        padded_inputs.append(padded_input)
        attention_masks.append(mask)

    padded_inputs = torch.stack(padded_inputs)
    targets = torch.stack(list(targets))
    attention_masks = torch.stack(attention_masks)

    return padded_inputs, targets, attention_masks

print("✓ Custom collate_fn defined for Character-Level Seq-to-One!")
print("\nThis function will:")
print("  1. Find the longest input character sequence in each batch")
print("  2. Pad shorter sequences with <PAD> tokens (index 0)")
print("  3. Create attention masks to mark real vs. padding characters")
print("  4. Stack target values (no padding needed — they're single characters!)")

**Creating the DataLoader**

The DataLoader batches our character sequences, shuffles them between epochs, and applies our padding function automatically.

In [ ]:
# Create DataLoader
batch_size = 64  # Larger batch size — character vocab is small so each forward pass is cheap

dataloader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn
)

print(f"✓ DataLoader created!")
print(f"\nDataLoader configuration:")
print(f"  - Batch size: {batch_size}")
print(f"  - Total batches: {len(dataloader):,}")
print(f"  - Shuffle: True")
print(f"  - Custom padding: Yes")
print(f"\nTotal samples seen in one epoch: {len(dataloader) * batch_size:,}")

**Visualizing a Batch**

Let's fetch one batch from the DataLoader and examine its structure.

In [ ]:
# Get a single batch from the DataLoader
batch_iterator = iter(dataloader)
inputs, targets, masks = next(batch_iterator)

print("✓ Successfully loaded a batch (Character-Level Seq-to-One)!\n")
print(f"Batch shapes:")
print(f"  - Inputs:  {inputs.shape}  (batch_size, sequence_length_in_chars)")
print(f"  - Targets: {targets.shape}  (batch_size,)  ← Just single char indices!")
print(f"  - Masks:   {masks.shape}  (batch_size, sequence_length_in_chars)")

print(f"\nBatch statistics:")
print(f"  - Batch size: {inputs.shape[0]}")
print(f"  - Input sequence length: {inputs.shape[1]} characters")
print(f"  - Total input chars: {inputs.numel():,}")
print(f"  - Padding chars: {(inputs == 0).sum().item():,}")
print(f"  - Real chars: {(inputs != 0).sum().item():,}")

In [ ]:
# Examine the first 3 sequences in the batch
print("Detailed view of first 3 sequences in the batch (Character-Level Seq-to-One):\n")

for i in range(min(3, inputs.shape[0])):
    print(f"Sequence {i+1}:")

    input_seq = inputs[i]
    target = targets[i]
    mask_seq = masks[i]

    real_length = mask_seq.sum().item()

    input_chars = [idx2char[idx.item()] for idx in input_seq[:real_length]]
    target_char = idx2char[target.item()]

    print(f"  Length: {real_length} characters")
    print(f"  Input:  {''.join(input_chars)!r}")
    print(f"  Target: {target_char!r}  ← Predict this ONE character")
    print()

### **5. Summary: The Complete Character-Level Data Pipeline**

You've built a complete character-level data pipeline. Let's recap:

**1. Text → Character Tokens**
- Simply split text into individual characters with `list(text)`
- No special rules for punctuation or whitespace
- Built char-to-index mappings with only ~68 unique characters

**2. Tokens → Character Sequences (Seq-to-One)**
- Sliding window of `seq_length = 100` characters
- Each sample: 100 input chars → 1 target char

**3. Sequences → Batches**
- Custom `collate_fn` pads variable-length sequences
- Attention masks mark real vs. padding characters
- Targets are 1D (one char index per sample)

**Word-Level vs Character-Level:**

| | **Word-Level** | **Character-Level** |
|---|---|---|
| Tokenizer | Split on whitespace + punctuation | `list(text)` |
| Vocab size | ~25,000 words | ~68 characters |
| Seq length | 20 words | 100 chars |
| OOV problem | Yes (unseen words) | No (all chars known) |
| Granularity | Coarse | Fine |

---

### 🎉 **Part 1 Complete!**

You've successfully built a character-level data preprocessing pipeline! You now understand:

✅ How character tokenization differs from word tokenization  
✅ Why character vocabularies are tiny (~68 chars vs ~25,000 words)  
✅ How to create training pairs using sliding windows at the char level  
✅ How to handle variable-length character sequences with padding  
✅ How to create efficient DataLoaders for character-level batch processing  

---

## **Part 2: The Engine – Building and Training the Character-Level RNN**

**Objective**
Construct a GRU-based recurrent neural network for **character-level** next-character prediction using the seq-to-one approach.

The architecture is nearly identical to the word-level RNN — the key changes are:
- Smaller `embedding_dim` (tiny vocabulary doesn't need large embeddings)
- Longer sequence processing (100 chars instead of 20 words)
- Character-level output (predicting one of ~68 chars)

---

### **6. Understanding the Character-Level RNN**

**How the Character-Level GRU works:**

At each character position $t$:
1. Embed the current character $c_t$ into a dense vector
2. Combine with previous hidden state $h_{t-1}$: $h_t = \text{GRU}(\text{embed}(c_t), h_{t-1})$
3. After processing all 100 input characters, take the **last hidden state** $h_{100}$
4. Map $h_{100}$ to a probability distribution over the 68 possible next characters

**The model must learn:**
- That certain letter combinations spell words ("th", "he", "ing")
- That words have typical endings ("-tion", "-ing", "-ed")
- The rhythmic patterns of Shakespearean verse
- Punctuation rules (spaces after commas, capitals after periods)

All from just predicting one character at a time!

### **7. Building the Character-Level RNN Architecture**

In [ ]:
class CharRNNPredictor(nn.Module):
    """
    Character-level GRU language model using Seq-to-One approach.
    Predicts the next character from the final hidden state of the GRU.
    """

    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers=2, dropout=0.3):
        """
        Initialize the character-level RNN model.

        Args:
            vocab_size: Number of unique characters (including special tokens)
            embedding_dim: Dimension of character embeddings
            hidden_dim: Dimension of the GRU hidden state
            num_layers: Number of stacked GRU layers
            dropout: Dropout rate for regularization
        """
        super(CharRNNPredictor, self).__init__()

        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        # Layer 1: Character Embedding
        # Smaller embedding_dim than word-level (vocab is tiny)
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)

        # Layer 2: GRU — processes the character sequence with memory
        self.gru = nn.GRU(
            embedding_dim,
            hidden_dim,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0,
            batch_first=True
        )

        self.dropout = nn.Dropout(dropout)

        # Layer 3: Output — maps ONLY the last hidden state to character logits
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden=None):
        """
        Forward pass (Character-Level Seq-to-One).

        Args:
            x: Input tensor of shape (batch_size, seq_length)
            hidden: Previous hidden state (optional)

        Returns:
            output: Logits of shape (batch_size, vocab_size) — ONE prediction per sequence
            hidden: Updated hidden state
        """
        # 1. Embed the input characters
        # (batch_size, seq_length) → (batch_size, seq_length, embedding_dim)
        embedded = self.embedding(x)

        # 2. Detach hidden state (Truncated BPTT)
        if hidden is not None:
            hidden = hidden.detach()

        # 3. Pass through GRU
        # (batch_size, seq_length, embedding_dim) → (batch_size, seq_length, hidden_dim)
        gru_out, hidden = self.gru(embedded, hidden)

        # 4. Take ONLY the last time step (seq-to-one!)
        # (batch_size, seq_length, hidden_dim) → (batch_size, hidden_dim)
        last_output = gru_out[:, -1, :]

        # 5. Apply dropout
        last_output = self.dropout(last_output)

        # 6. Map to vocabulary logits
        # (batch_size, hidden_dim) → (batch_size, vocab_size)
        output = self.fc(last_output)

        return output, hidden

    def init_hidden(self, batch_size):
        """Initialize hidden state with zeros."""
        return torch.zeros(self.num_layers, batch_size, self.hidden_dim).to(device)


# Hyperparameters
# Smaller embedding_dim is fine — only ~68 chars to distinguish
embedding_dim = 64
hidden_dim = 512   # Larger hidden — model must learn spelling + grammar from scratch
num_layers = 2
dropout = 0.3

model = CharRNNPredictor(vocab_size, embedding_dim, hidden_dim, num_layers, dropout).to(device)

print("✓ Character-Level RNN Model created (Seq-to-One)!")
print(f"\nModel architecture:")
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"\nVocab size (characters): {vocab_size}")
print(f"Embedding dim: {embedding_dim}  (small — tiny vocab)")
print(f"Hidden dim: {hidden_dim}  (large — must learn spelling + grammar)")

### **8. Setting Up Training**

In [ ]:
# Loss function and optimizer
criterion = nn.CrossEntropyLoss(ignore_index=0)  # Ignore padding (index 0)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print("✓ Training setup complete!")
print(f"\nLoss function: CrossEntropyLoss (ignoring <PAD> characters)")
print(f"Optimizer: Adam  (lr=0.001)")
print(f"\nReady to train the character-level language model!")

### **9. The Training Loop**

In [ ]:
def train_model(model, dataloader, criterion, optimizer, num_epochs=10):
    """
    Train the character-level RNN model using Seq-to-One approach.

    Args:
        model: The CharRNNPredictor model
        dataloader: DataLoader providing character-level batches
        criterion: Loss function (CrossEntropyLoss)
        optimizer: Adam optimizer
        num_epochs: Number of training epochs

    Returns:
        loss_history: List of average losses per epoch
    """
    model.train()
    loss_history = []

    print("Starting character-level training (Seq-to-One)...\n")
    print(f"{'Epoch':<8} {'Batch':<10} {'Loss':<12} {'Time':<10}")
    print("-" * 45)

    for epoch in range(num_epochs):
        epoch_start_time = time.time()
        total_loss = 0

        for batch_idx, (inputs, targets, masks) in enumerate(dataloader):
            hidden = None

            inputs = inputs.to(device)
            targets = targets.to(device)  # Shape: (batch_size,)

            optimizer.zero_grad()

            # Forward pass
            # outputs shape: (batch_size, vocab_size)  ← one prediction per sequence
            # targets shape: (batch_size,)             ← one char per sequence
            outputs, hidden = model(inputs, hidden)

            # Simple loss — no reshaping needed (seq-to-one advantage!)
            loss = criterion(outputs, targets)

            loss.backward()

            # Gradient clipping for stability
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)

            optimizer.step()

            total_loss += loss.item()

            if (batch_idx + 1) % 500 == 0:
                avg_loss = total_loss / (batch_idx + 1)
                elapsed = time.time() - epoch_start_time
                print(f"{epoch+1:<8} {batch_idx+1:<10} {avg_loss:<12.4f} {elapsed:<10.2f}s")

        avg_epoch_loss = total_loss / len(dataloader)
        loss_history.append(avg_epoch_loss)

        epoch_time = time.time() - epoch_start_time
        print(f"Epoch {epoch+1} complete - Avg Loss: {avg_epoch_loss:.4f} - Time: {epoch_time:.2f}s")
        print("-" * 45)

    print("\n✓ Character-level training complete!")
    return loss_history


# Train the model
num_epochs = 10
loss_history = train_model(model, dataloader, criterion, optimizer, num_epochs)

In [ ]:
# Plot the training loss
plt.figure(figsize=(10, 6))
plt.plot(range(1, num_epochs + 1), loss_history, marker='o', linewidth=2, markersize=8, color='#E67E22')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Average Loss', fontsize=12)
plt.title('Character-Level RNN: Training Loss Over Time', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nFinal training loss: {loss_history[-1]:.4f}")
print(f"Loss improvement: {loss_history[0] - loss_history[-1]:.4f}")
print(f"\nNote: Character-level loss tends to be lower than word-level")
print(f"because the vocabulary is much smaller (~68 vs ~25,000 options).")

### **10. Saving the Trained Model**

In [ ]:
# Save the model checkpoint
checkpoint = {
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'vocab_size': vocab_size,
    'embedding_dim': embedding_dim,
    'hidden_dim': hidden_dim,
    'num_layers': num_layers,
    'char2idx': char2idx,
    'idx2char': idx2char,
    'loss_history': loss_history
}

checkpoint_path = 'shakespeare_char_rnn_model.pth'
torch.save(checkpoint, checkpoint_path)

print(f"✓ Character-level model saved to {checkpoint_path}")
print(f"\nCheckpoint contains:")
print(f"  - Model weights (char-level seq-to-one GRU)")
print(f"  - Optimizer state")
print(f"  - Character vocabulary ({vocab_size} tokens)")
print(f"  - Training history ({len(loss_history)} epochs)")

### 🎉 **Part 2 Complete!**

You've successfully trained a character-level seq-to-one RNN! You now understand:

✅ How the architecture adapts from word-level to character-level  
✅ Why we use a larger hidden_dim for character models  
✅ How the training loop is identical — only the data changes  
✅ The computational advantages of a tiny vocabulary  

---

## **Part 3: The Critique – Generation and Analysis**

**Objective**
Generate Shakespeare-style text **character by character** using the trained model.

---

### **11. Loading the Trained Model**

In [ ]:
# Redefine the model class for loading
class CharRNNPredictor(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers=2, dropout=0.3):
        super(CharRNNPredictor, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.gru = nn.GRU(embedding_dim, hidden_dim, num_layers=num_layers,
                          dropout=dropout if num_layers > 1 else 0, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden=None):
        embedded = self.embedding(x)
        if hidden is not None:
            hidden = hidden.detach()
        gru_out, hidden = self.gru(embedded, hidden)
        last_output = gru_out[:, -1, :]
        last_output = self.dropout(last_output)
        output = self.fc(last_output)
        return output, hidden

    def init_hidden(self, batch_size):
        return torch.zeros(self.num_layers, batch_size, self.hidden_dim).to(device)

print("✓ Model class defined")

In [ ]:
# Load the saved checkpoint
checkpoint_path = 'shakespeare_char_rnn_model.pth'

try:
    checkpoint = torch.load(checkpoint_path, map_location=device)
    print(f"✓ Loaded checkpoint from {checkpoint_path}")

    vocab_size = checkpoint['vocab_size']
    embedding_dim = checkpoint['embedding_dim']
    hidden_dim = checkpoint['hidden_dim']
    num_layers = checkpoint['num_layers']
    char2idx = checkpoint['char2idx']
    idx2char = checkpoint['idx2char']

    print(f"\nModel configuration (Character-Level Seq-to-One):")
    print(f"  - Vocabulary size: {vocab_size} characters")
    print(f"  - Embedding dimension: {embedding_dim}")
    print(f"  - Hidden dimension: {hidden_dim}")
    print(f"  - Number of layers: {num_layers}")

except FileNotFoundError:
    print(f"❌ Model file '{checkpoint_path}' not found!")
    print("Please run Part 2 to train and save the model first.")

In [ ]:
# Recreate model and load weights
model = CharRNNPredictor(vocab_size, embedding_dim, hidden_dim, num_layers).to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print("✓ Character-level model loaded and ready for inference!")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

### **12. Character-Level Text Generation**

**How Character-Level Generation Works**

The auto-regressive process is the same as word-level, but at each step we predict and append a **single character**:

1. Start with a seed string (e.g., `"To be or not to"`)
2. Feed the last `seq_length` characters through the model
3. Sample the next character from the predicted distribution
4. Append it to the sequence
5. Repeat until the desired length is reached

**Key difference from word-level generation:**
- No space-handling logic needed — characters are joined directly
- We generate many more steps to produce readable text (e.g., 500 characters ≈ ~80 words)
- Temperature still controls creativity

In [ ]:
def generate_text(model, seed_text, char2idx, idx2char, max_length=500, temperature=1.0):
    """
    Generate text character-by-character using the trained character-level RNN.

    Args:
        model: Trained CharRNNPredictor model
        seed_text: Starting string to continue from
        char2idx: Dictionary mapping characters to indices
        idx2char: Dictionary mapping indices to characters
        max_length: Number of NEW characters to generate
        temperature: Controls randomness (lower = more conservative, higher = more creative)

    Returns:
        generated_text: The seed + generated characters as a string
    """
    model.eval()

    # Convert seed text to character indices
    input_seq = [char2idx.get(c, char2idx['<UNK>']) for c in seed_text]

    hidden = model.init_hidden(1)

    with torch.no_grad():
        for _ in range(max_length):
            x = torch.tensor([input_seq], dtype=torch.long).to(device)

            output, hidden = model(x, hidden)

            # Get logits for the predicted next character
            logits = output[0]  # shape: (vocab_size,)

            # Apply temperature scaling
            logits = logits / temperature

            # Sample from the distribution
            probs = torch.softmax(logits, dim=0)
            next_char_idx = torch.multinomial(probs, 1).item()

            # Skip padding token
            if next_char_idx == 0:
                continue

            # Stop at end-of-sequence
            if next_char_idx == char2idx.get('<EOS>', -1):
                break

            # Append the new character
            input_seq.append(next_char_idx)

    # Convert all indices back to characters and join (no spaces needed!)
    return ''.join(idx2char[idx] for idx in input_seq)

print("✓ Character-level text generation function ready!")
print("\nNote: Characters are joined directly — no space logic needed.")

### **13. Testing the Model: Shakespeare-Style Character Generation**

In [ ]:
# Test with different seed texts and temperatures
seed_texts = [
    "To be or not to be",
    "The king said",
    "What is thy name",
    "Good morning"
]

temperatures = [0.5, 1.0, 1.5]

print("=" * 80)
print("CHARACTER-LEVEL GENERATED TEXT SAMPLES")
print("=" * 80)

for seed in seed_texts:
    print(f"\n{'='*80}")
    print(f"SEED: {seed!r}")
    print(f"{'='*80}\n")

    for temp in temperatures:
        print(f"Temperature = {temp}:")
        print("-" * 80)
        generated = generate_text(model, seed, char2idx, idx2char, max_length=300, temperature=temp)
        print(generated)
        print()

### **14. Analysing the Generated Text**

**What can a character-level RNN learn?** ✅

1. **Spelling** — The model learns that `t` is often followed by `h`, that `ing` is a common ending, etc.
2. **Words** — After training, most generated "words" are real English words
3. **Punctuation rules** — Spaces after commas, capitals at line starts
4. **Shakespearean style** — Archaic words (`thou`, `thy`, `hath`) should emerge
5. **Verse structure** — Line breaks at roughly the right places

**Comparing Temperature Effects:**
- `T = 0.5`: More repetitive but consistently valid words and grammar
- `T = 1.0`: Good balance — mostly real words with creative combinations
- `T = 1.5`: More novel character sequences, some gibberish

**Limitations (same as word-level, but amplified):**
- Long-range dependencies are even harder (100+ character context needed for meaning)
- The hidden state bottleneck is more severe at the character level
- This motivates attention mechanisms even more strongly!

In [ ]:
# Analyse character frequency in generated text vs. training data
generated_sample = generate_text(
    model, "First Citizen:\n", char2idx, idx2char, max_length=1000, temperature=1.0
)

print("=" * 80)
print("LONG GENERATED SAMPLE (1000 characters, temperature=1.0):")
print("=" * 80)
print(generated_sample)
print("=" * 80)

# Count character frequencies
gen_chars = Counter(generated_sample)
orig_chars = Counter(text[:10000])  # Sample of original

# Compare the most common characters
print("\nTop 10 most frequent characters:")
print(f"{'Character':<12} {'Generated %':<15} {'Original %':<15}")
print("-" * 42)
top_chars = [c for c, _ in orig_chars.most_common(10)]
gen_total = sum(gen_chars.values())
orig_total = sum(orig_chars.values())
for c in top_chars:
    gen_pct = 100 * gen_chars.get(c, 0) / gen_total
    orig_pct = 100 * orig_chars.get(c, 0) / orig_total
    print(f"{repr(c):<12} {gen_pct:<15.2f} {orig_pct:<15.2f}")

### **15. Visualising the Long-Range Dependency Bottleneck**

In [ ]:
# Visualize information flow in character-level RNN
fig, ax = plt.subplots(figsize=(14, 6))

# Character-level sequences are MUCH longer than word-level
time_steps = np.arange(1, 201)

# Information decay (faster at char level — more steps to cover same semantic distance)
char_rnn_decay = 100 * np.exp(-0.04 * time_steps)
word_rnn_decay = 100 * np.exp(-0.07 * time_steps)  # Faster because words are coarser
ideal = np.ones_like(time_steps) * 95

ax.plot(time_steps, char_rnn_decay, linewidth=3, color='#E74C3C', label='Char-Level RNN')
ax.plot(time_steps, word_rnn_decay, linewidth=3, color='#F39C12', label='Word-Level RNN', linestyle='--')
ax.plot(time_steps, ideal, linewidth=3, color='#3498DB', label='Ideal (Transformers)', linestyle=':')
ax.fill_between(time_steps, 0, char_rnn_decay, alpha=0.15, color='#E74C3C')

ax.axvline(x=100, color='gray', linestyle='--', alpha=0.5, label='Seq length = 100 chars')
ax.set_xlabel('Position (Character Steps)', fontsize=12, fontweight='bold')
ax.set_ylabel('Information Retained (%)', fontsize=12, fontweight='bold')
ax.set_title('Character-Level RNN: Information Decay vs. Alternatives', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11)
ax.set_ylim([0, 105])

# Annotate
ax.annotate('Our seq_length=100\n(context window)', xy=(100, 50), xytext=(120, 70),
            arrowprops=dict(arrowstyle='->', color='gray'),
            fontsize=10, bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("KEY INSIGHT: Character vs Word-Level RNNs")
print("="*80)
print("\n📉 Char-level RNN: Decays slower per step (chars are small)")
print("   BUT needs many more steps to span the same semantic distance")
print("   → 100 chars ≈ 15-20 words. The model sees less context!")
print("\n💡 This motivates both longer sequences AND attention mechanisms.")
print("   Transformers solve this by attending to ALL positions at once!")
print("="*80)

### **16. Summary: Word-Level vs Character-Level**

🎉 **Congratulations!** You've completed the full character-level seq-to-one RNN pipeline.

**Complete Comparison: Word-Level vs Character-Level Seq-to-One RNN**

| **Aspect** | **Word-Level** | **Character-Level** |
|---|---|---|
| Tokenizer | Split on whitespace/punctuation | `list(text)` |
| Vocab size | ~25,000 words | ~68 characters |
| Seq length | 20 tokens | 100 tokens |
| Embedding dim | 128 | 64 |
| Hidden dim | 256 | 512 |
| OOV problem | Yes | No |
| Generation step | One word | One character |
| Chars to generate 100 words | ~500 steps | ~500 steps (direct) |
| Can spell novel words | No | Yes |
| Training stability | Good | Good |

**When to Use Character-Level Models:**
- ✅ Morphologically rich languages (many word forms)
- ✅ When vocabulary is dynamic (names, codes, URLs)
- ✅ Spelling correction tasks
- ✅ When memory is constrained (tiny vocab = small embedding table)

**When to Use Word-Level Models:**
- ✅ Standard NLP tasks with fixed vocabulary
- ✅ When semantic meaning per token is important
- ✅ Faster training convergence (more information per token)

**Both share the same fundamental RNN bottleneck** — this motivates **Transformers and Attention**, coming in the next days!

---

**Key Takeaways:**
- Character-level models require only trivial tokenization
- Tiny vocabulary eliminates OOV problems entirely
- The same seq-to-one architecture works for both granularities
- Longer sequences are needed to capture the same semantic context
- Temperature sampling controls creativity at both levels